In [ ]:
import sys
import csv
import importlib
from pathlib import Path

import numpy as np
import pandas as pd

cwd = Path.cwd().resolve()
repo_root = next((c for c in [cwd, *cwd.parents] if (c / "utilities" / "functions.py").exists()), None)
if repo_root is None:
    raise FileNotFoundError("Could not find the repository root containing utilities/functions.py")
sys.path.append(str(repo_root))

import utilities.functions as functions

importlib.reload(functions)

data_root = repo_root / "ms0_5"

SEQS = {'IN': functions.read_seq(str(data_root / "IN" / "data" / "in.reduce4.seq")),
        'PR': functions.read_seq(str(data_root / "PR" / "data" / "pr.exper.reduce4.seq")),
        'RT': functions.read_seq(str(data_root / "RT" / "data" / "rt.reduce4.seq"))}
REDUX = {'IN': functions.get_redu_dict(str(data_root / "IN" / "data" / "in.reduce4.redux"), 1),
         'PR': functions.get_redu_dict(str(data_root / "PR" / "data" / "pr.reduce4.redux"), 0),
         'RT': functions.get_redu_dict(str(data_root / "RT" / "data" / "rt.reduce4.redux"), 0)}
RANGE = {'IN': (1, 263), 'PR': (1, 99), 'RT': (39, 226)}


def build_J_matrix(j_file, min_position, max_position):
    """Dense coupling tensor, shape (L, L, 4, 5), indexed [p1, p2, aa_at_p1, aa_at_p2]."""
    J = np.load(j_file).astype(np.float32)
    L = max_position - min_position + 1
    Jm = np.zeros((L, L, 4, 5), dtype=np.float32)
    iu0, iu1 = np.triu_indices(L, 1)
    blocks = J.reshape(-1, 4, 4)
    assert blocks.shape[0] == iu0.size, "J.npy row count does not match the position range"
    Jm[iu0, iu1, :, :4] = blocks
    Jm[iu1, iu0, :, :4] = blocks.transpose(0, 2, 1)
    return Jm


JS = {'IN': build_J_matrix(str(data_root / "IN" / "data" / "J.npy"), 1, 263),
      'PR': build_J_matrix(str(data_root / "PR" / "data" / "J_PR.npy"), 1, 99),
      'RT': build_J_matrix(str(data_root / "RT" / "data" / "J_RT.npy"), 39, 226)}

print({p: f"{len(s):,} sequences" for p, s in SEQS.items()})

In [ ]:
# ---------------------------------------------------------------------------
# dE double per sequence, for one pair. Same construction as everywhere else: the sequence is
# rewritten to wild type at both positions, and
#     dE double = M[wt1, wt2] - M[mt1, mt2],   M[a1, a2] = S(p1, a1) + S(p2, a2) + J[p1, p2, a1, a2]
# ---------------------------------------------------------------------------
_AA_CODE = np.full(256, 4, dtype=np.uint8)
for _i, _c in enumerate("ABCD"):
    _AA_CODE[ord(_c)] = _i


def encode_seqs(seq_list, min_pos, max_pos):
    L = max_pos - min_pos + 1
    N = len(seq_list)
    raw = np.frombuffer("".join(seq_list).encode(), dtype=np.uint8)
    assert raw.size == N * L, "all sequences must span exactly min_pos..max_pos"
    codes = _AA_CODE[raw.reshape(N, L)]
    onehot = np.zeros((N * L, 5), dtype=np.float32)
    onehot[np.arange(N * L), codes.ravel()] = 1.0
    return codes, onehot.reshape(N, L * 5)


def _site_energies(onehot, Jm, p, excluded):
    A = Jm[p].copy()
    A[list(excluded)] = 0.0
    return np.asarray(onehot @ A.transpose(0, 2, 1).reshape(-1, 4), dtype=np.float64)


def de_double(onehot, Jm, p1i, p2i, wt1, mt1, wt2, mt2):
    S1 = _site_energies(onehot, Jm, p1i, (p1i, p2i))
    S2 = _site_energies(onehot, Jm, p2i, (p1i, p2i))
    J12 = Jm[p1i, p2i, :, :4].astype(np.float64)
    M = S1[:, :, None] + S2[:, None, :] + J12[None, :, :]
    return M[:, wt1, wt2] - M[:, mt1, mt2]


def spearman(x, y):
    """Rank correlation, without pulling in scipy: Pearson r of the ranks."""
    rx = np.argsort(np.argsort(x)).astype(float)
    ry = np.argsort(np.argsort(y)).astype(float)
    return float(np.corrcoef(rx, ry)[0, 1])


print("machinery ready")

In [ ]:
# =============================================================================
# The two numbers per pair.
#
# x  how much the background moves the pair: the 5th-95th percentile spread of dE double over
#    every sequence in the alignment. This is the width of the grey bar in the
#    consensus-vs-realized figure, computed here rather than read from anywhere.
#
# y  where that pair's carriers end up: of the sequences that CARRY the pair, the share
#    whose background the model calls gain of fitness -- (gof and carrier) / carrier. This is
#    not the ratio the observed-vs-expected figure plots, which divides the same numerator by
#    the number of gain-of-fitness backgrounds instead: "given the pair is here, was the
#    background favourable?" against "given the background is favourable, is the pair here?".
#    Both come out of the probability CSVs (num_with_DMC per category, and the pair's Total).
#
# Pairs are the ones in those figures, all three proteins, so the switches below have to match
# the run that produced the CSVs.
# =============================================================================
DMC_MIN_PERCENT = {'IN': 1.0, 'PR': 5.0, 'RT': 5.0}
TOP_N_PAIRS = 20
CSV_VERSION = 'v17'
USE_TOP_DDE_PAIRS = True
NON_OVERLAPPING = False        # False -> the "_with_overlap" CSVs

STEMS = {'IN': ('integrase_all_probabilities', 'INSTI'),
         'PR': ('protease_all_probabilities', 'PI'),
         'RT': ('reverseTranscriptase_both_probabilities', 'NRTI/NNRTI')}


def csv_name(stem, protein):
    tags = [f"{stem}_{CSV_VERSION}"]
    if USE_TOP_DDE_PAIRS:
        tags.append(f"topdde_{DMC_MIN_PERCENT[protein]:g}pct")
        if TOP_N_PAIRS is not None:
            tags.append(f"top{TOP_N_PAIRS}")
    if not NON_OVERLAPPING:
        tags.append("with_overlap")
    return "_".join(tags) + ".csv"


rows = []
for prot, (stem, drug) in STEMS.items():
    df = pd.read_csv(csv_name(stem, prot))
    gof = df[df['epistasis_subset'] == 'Gain_of_function'].set_index('mutation_pair')
    total = df[df['epistasis_subset'] == 'Total'].set_index('mutation_pair')

    mn, mx = RANGE[prot]
    codes, onehot = encode_seqs(SEQS[prot], mn, mx)
    for name in gof.index:
        a, b = functions.split_pairs(name)
        wt1, pos1, mt1 = functions.split_pair(functions.unreduced_to_reduced(REDUX[prot], a))
        wt2, pos2, mt2 = functions.split_pair(functions.unreduced_to_reduced(REDUX[prot], b))
        idx = (pos1 - mn, pos2 - mn, "ABCD".index(wt1), "ABCD".index(mt1),
               "ABCD".index(wt2), "ABCD".index(mt2))
        de = de_double(onehot, JS[prot], *idx)
        lo, hi = np.percentile(de, [5, 95])
        n_carriers = int(total.loc[name, 'num_with_DMC'])
        rows.append({'protein': prot, 'drug': drug, 'pair': name,
                     'dE double range': hi - lo,
                     'dE double sd': de.std(),
                     'dE double median': np.median(de),
                     'carriers': n_carriers,
                     'gof carriers': int(gof.loc[name, 'num_with_DMC']),
                     '% of carriers with gof background': 100 * gof.loc[name, 'num_with_DMC'] / n_carriers})

PAIRS = pd.DataFrame(rows).sort_values('dE double range', ascending=False).reset_index(drop=True)
PAIRS.to_csv('de_range_vs_gof_share.csv', index=False)
print(f"{len(PAIRS)} pairs   ->  de_range_vs_gof_share.csv")
PAIRS.head(10).round(2)

In [ ]:
# =============================================================================
# The correlation, pooled and per protein. Pearson asks whether the relationship is linear,
# Spearman only whether it is monotone -- worth both, since the share is a percentage and is
# bounded at 100, which flattens the top of any linear fit.
# =============================================================================
x, y = PAIRS['dE double range'].values, PAIRS['% of carriers with gof background'].values
print(f"pooled ({len(PAIRS)} pairs):   Pearson r = {np.corrcoef(x, y)[0, 1]:.3f}   "
      f"Spearman rho = {spearman(x, y):.3f}")
for prot, g in PAIRS.groupby('protein'):
    xa, ya = g['dE double range'].values, g['% of carriers with gof background'].values
    print(f"  {prot} ({len(g):2d} pairs):  Pearson r = {np.corrcoef(xa, ya)[0, 1]:6.3f}   "
          f"Spearman rho = {spearman(xa, ya):6.3f}")

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# =============================================================================
# The scatter: one point per pair, x its dE double range, y the share of its carriers sitting
# in gain-of-fitness backgrounds. Colour is the protein, marker area is the number of carriers
# -- a percentage from 13 carriers and one from 6,000 are not the same evidence, and IN's
# pairs are all in the first group. The grey line is the pooled least-squares fit; the pairs
# furthest from it are labelled.
# =============================================================================
SURFACE, INK, INK2, MUTED, GRID, BASE = '#fcfcfb', '#0b0b0b', '#52514e', '#898781', '#e6e5df', '#c3c2b7'
DRUG_COLOR = {'INSTI': '#1f77b4', 'PI': '#ff7f0e', 'NRTI/NNRTI': '#2ca02c'}

fig, ax = plt.subplots(figsize=(6.6, 4.6))
for drug, g in PAIRS.groupby('drug'):
    ax.scatter(g['dE double range'], g['% of carriers with gof background'],
               s=g['carriers'] / 6000 * 130 + 14, color=DRUG_COLOR[drug], alpha=0.65,
               edgecolors='black', linewidth=0.35, zorder=3)

slope, intercept = np.polyfit(x, y, 1)
xs = np.linspace(x.min(), x.max(), 2)
ax.plot(xs, slope * xs + intercept, color=BASE, lw=1.0, zorder=2)

resid = y - (slope * x + intercept)
for i in np.argsort(-abs(resid))[:5]:
    r = PAIRS.iloc[i]
    ax.annotate(r['pair'], (r['dE double range'], r['% of carriers with gof background']),
                textcoords='offset points', xytext=(7, -1), fontsize=7.5, color=INK2)

ax.set_xlabel('dE double range across backgrounds  (95th − 5th percentile)', fontsize=9,
              color=INK2)
ax.set_ylabel('share of the pair’s carriers whose\nbackground is gain of fitness  (%)',
              fontsize=9, color=INK2)
ax.set_ylim(-4, 104)
ax.tick_params(length=0, labelsize=8)
for s in ('top', 'right'):
    ax.spines[s].set_visible(False)
for s in ('left', 'bottom'):
    ax.spines[s].set_color(BASE)
ax.grid(color=GRID, linewidth=0.6)
ax.set_axisbelow(True)
ax.legend(handles=[Line2D([], [], marker='o', color=c, lw=0, markersize=5,
                          markeredgecolor='black', markeredgewidth=0.35, label=d)
                   for d, c in DRUG_COLOR.items()],
          loc='lower left', bbox_to_anchor=(0, 1.0), ncol=3, frameon=False, fontsize=8,
          labelcolor=INK2, handletextpad=0.35, columnspacing=1.2)
ax.text(1.0, 1.0, f'r = {np.corrcoef(x, y)[0, 1]:.2f}   ρ = {spearman(x, y):.2f}',
        transform=ax.transAxes, ha='right', va='bottom', fontsize=8.5, color=INK2)

fig.savefig('de_range_vs_gof_share.png', dpi=400, bbox_inches='tight', pad_inches=0.02,
            facecolor=SURFACE)
fig.savefig('de_range_vs_gof_share.pdf', bbox_inches='tight', pad_inches=0.02,
            facecolor=SURFACE)
print('saved de_range_vs_gof_share.png / .pdf')
plt.show()

In [ ]:
# =============================================================================
# The same relationship, one panel per protein.
#
# Pooling the three hides what is going on: each protein sits in its own band of the plot, so
# a single fit through all 56 pairs is flatter than the fit inside any one of them (r = 0.59
# pooled against 0.84 and 0.70 for PR and RT). Panels share the axes, so the bands are still
# comparable, and each carries its own least-squares line with its own r and rho.
#
# IN is the honest counter-example and is left in: 20 pairs whose carrier counts run 13-221,
# where the share is quantised in steps of several percent and no trend survives.
# =============================================================================
fig, axes = plt.subplots(1, 3, figsize=(10.6, 3.8), sharex=True, sharey=True,
                         layout='constrained')

for ax, (prot, g) in zip(axes, PAIRS.groupby('protein')):
    drug = g['drug'].iloc[0]
    xa, ya = g['dE double range'].values, g['% of carriers with gof background'].values
    ax.scatter(xa, ya, s=g['carriers'] / 6000 * 130 + 14, color=DRUG_COLOR[drug], alpha=0.65,
               edgecolors='black', linewidth=0.35, zorder=3)

    sl, ic = np.polyfit(xa, ya, 1)
    xs = np.linspace(xa.min(), xa.max(), 2)
    ax.plot(xs, sl * xs + ic, color=BASE, lw=1.0, zorder=2)

    res = ya - (sl * xa + ic)
    for i in np.argsort(-abs(res))[:2]:
        ax.annotate(g['pair'].iloc[i], (xa[i], ya[i]), textcoords='offset points',
                    xytext=(6, -1), fontsize=7, color=INK2)

    ax.set_title(f'{prot}  ({drug}),  {len(g)} pairs', fontsize=9, color=INK, loc='left',
                 pad=6)
    ax.text(0.97, 0.05, f'r = {np.corrcoef(xa, ya)[0, 1]:.2f}\nρ = {spearman(xa, ya):.2f}',
            transform=ax.transAxes, ha='right', va='bottom', fontsize=8.5, color=INK2,
            linespacing=1.4)
    ax.tick_params(length=0, labelsize=8)
    for s in ('top', 'right'):
        ax.spines[s].set_visible(False)
    for s in ('left', 'bottom'):
        ax.spines[s].set_color(BASE)
    ax.grid(color=GRID, linewidth=0.6)
    ax.set_axisbelow(True)

axes[0].set_ylim(-6, 106)
fig.supxlabel('dE double range across backgrounds  (95th − 5th percentile)', fontsize=9,
              color=INK2)
fig.supylabel('share of carriers whose background is gain of fitness  (%)', fontsize=9,
              color=INK2)

fig.savefig('de_range_vs_gof_share_by_protein.png', dpi=400, bbox_inches='tight',
            pad_inches=0.02, facecolor=SURFACE)
fig.savefig('de_range_vs_gof_share_by_protein.pdf', bbox_inches='tight', pad_inches=0.02,
            facecolor=SURFACE)
print('saved de_range_vs_gof_share_by_protein.png / .pdf')
plt.show()